In [10]:
# ==============================================================================
# NOTEBOOK 02: PIPELINE ETL + TRACK 1 (MARKET BASKET ANALYSIS) - TODAS LAS REGLAS 1 A 1
# ==============================================================================

import time
import psutil
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías para Reglas de Asociación
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

print("✓ Librerías analíticas y de minería de datos cargadas correctamente.")

# 1. CARGA DE DATOS DESDE LA RUTA PROCESADA (GENERADOS EN EL EDA)
root = Path.cwd()
while not (root / 'data' / 'processed').is_dir():
    if root.parent == root:
        raise FileNotFoundError('No se encontró la carpeta data/processed del proyecto.')
    root = root.parent
processed_dir = root / 'data' / 'processed'

print("\n--- Cargando Datasets Limpios desde el Pipeline de EDA ---")
df_inv = pd.read_csv(processed_dir / 'INVENTARIO_PRODUCTOS_LIMPIO.csv')
df_ventas = pd.read_csv(processed_dir / 'ventas_limpia.csv')

# Tipificación de fechas en ventas
if 'fecha' in df_ventas.columns:
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'], errors='coerce')

# Tratamiento financiero seguro usando los nombres de columnas limpios del inventario
df_inv['costo_c(COP)'] = pd.to_numeric(df_inv['costo_c(COP)'], errors='coerce').replace(0, np.nan).fillna(0.01)
df_inv['precio_v(COP)'] = pd.to_numeric(df_inv['precio_v(COP)'], errors='coerce').replace(0, np.nan).fillna(0.01)
df_inv['stock'] = pd.to_numeric(df_inv['stock'], errors='coerce').fillna(0)
df_inv['MARGEN_BRUTO_PCT'] = ((df_inv['precio_v(COP)'] - df_inv['costo_c(COP)']) / df_inv['precio_v(COP)']) * 100

# Asegurar tipos de datos para los cruces de la matriz transaccional con nombres limpios
df_ventas['factura'] = df_ventas['factura'].astype(str)
df_ventas['codigo'] = df_ventas['codigo'].astype(str)
df_inv['codigo'] = df_inv['codigo'].astype(str)

# Construcción de la matriz transaccional (nivel factura-categoría)
inventario_categorias = df_inv[['codigo', 'categoria']].rename(
    columns={'categoria': 'CATEGORIA_PRODUCTO'}
).copy()

inventario_categorias['CATEGORIA_PRODUCTO'] = inventario_categorias['CATEGORIA_PRODUCTO'].replace(
    {'NAN': 'OTROS', 'NONE': 'OTROS', '<NA>': 'OTROS', '': 'OTROS'}
).fillna('OTROS')

inventario_categorias = (
    inventario_categorias.groupby('codigo')['CATEGORIA_PRODUCTO']
    .agg(lambda categorias: categorias.iloc[0] if categorias.nunique() == 1 else 'OTROS')
    .rename('CATEGORIA_PRODUCTO')
    .reset_index()
)

ventas_categoria = df_ventas.merge(
    inventario_categorias,
    on='codigo',
    how='left',
    validate='many_to_one'
)
ventas_categoria['CATEGORIA_PRODUCTO'] = ventas_categoria['CATEGORIA_PRODUCTO'].fillna('OTROS')

categorias_analisis = {'FILTROS', 'ACEITES', 'LLANTAS', 'BATERIAS'}
ventas_categoria = ventas_categoria[
    ventas_categoria['CATEGORIA_PRODUCTO'].isin(categorias_analisis)
].copy()

basket_raw = (ventas_categoria.groupby(['factura', 'CATEGORIA_PRODUCTO'])['cantidad']
              .sum().unstack(fill_value=0))

basket_sets = basket_raw.gt(0).astype(int)
print(f"✓ Matriz Transaccional lista: {basket_sets.shape[0]} facturas x {basket_sets.shape[1]} categorías.")

# ------------------------------------------------------------------------------
# 2. TRACK 1: BENCHMARKING DE REGLAS DE ASOCIACIÓN (FILTRADO 1 A 1)
# ------------------------------------------------------------------------------
print("\n--- Ejecutando Track 1: Market Basket Analysis (Relaciones 1 a 1) ---")

MIN_SUPP = 0.015  # Umbral de soporte mínimo
MIN_CONF = 0.15   # Umbral de confianza mínimo

# Función para asegurar que el antecedente y el consecuente tengan longitud igual a 1
def filtrar_reglas_1a1(df_rules):
    if df_rules.empty:
        return df_rules
    return df_rules[
        (df_rules['antecedents'].apply(lambda x: len(x) == 1)) & 
        (df_rules['consequents'].apply(lambda x: len(x) == 1))
    ].copy()

# Modelo 1: Apriori
t0 = time.time()
freq_apriori = apriori(basket_sets, min_support=MIN_SUPP, use_colnames=True)
rules_apriori_raw = association_rules(freq_apriori, metric="confidence", min_threshold=MIN_CONF)
rules_apriori = filtrar_reglas_1a1(rules_apriori_raw)
t_apriori = time.time() - t0

# Modelo 2: FP-Growth (Recomendado por velocidad y uso de memoria)
t0 = time.time()
freq_fpgrowth = fpgrowth(basket_sets, min_support=MIN_SUPP, use_colnames=True)
rules_fpgrowth_raw = association_rules(freq_fpgrowth, metric="confidence", min_threshold=MIN_CONF)
rules_fpgrowth = filtrar_reglas_1a1(rules_fpgrowth_raw)
t_fpgrowth = time.time() - t0

# Modelo 3: ECLAT (minería vertical mediante intersección de TID-sets)
t0 = time.time()
total_transacciones = len(basket_sets)
tidsets = {
    categoria: set(np.flatnonzero(basket_sets[categoria].to_numpy(dtype=bool)))
    for categoria in basket_sets.columns
}
itemsets_eclat = []

def minar_eclat(prefijo, candidatos, tidset_prefijo):
    for indice, item in enumerate(candidatos):
        tidset_item = tidset_prefijo & tidsets[item]
        soporte = len(tidset_item) / total_transacciones
        if soporte >= MIN_SUPP:
            itemset = prefijo | {item}
            itemsets_eclat.append({'support': soporte, 'itemsets': itemset})
            minar_eclat(itemset, candidatos[indice + 1:], tidset_item)

minar_eclat(set(), list(basket_sets.columns), set(range(total_transacciones)))
freq_eclat = pd.DataFrame(itemsets_eclat, columns=['support', 'itemsets'])
rules_eclat_raw = association_rules(freq_eclat, metric="confidence", min_threshold=MIN_CONF)
rules_eclat = filtrar_reglas_1a1(rules_eclat_raw)
t_eclat = time.time() - t0

# Resumen comparativo de rendimiento
bench_df = pd.DataFrame({
    'Algoritmo': ['Apriori', 'FP-Growth', 'ECLAT'],
    'Tiempo Cómputo (s)': [t_apriori, t_fpgrowth, t_eclat],
    'Reglas 1-a-1 Generadas': [len(rules_apriori), len(rules_fpgrowth), len(rules_eclat)],
    'Max Lift': [
        rules_apriori['lift'].max() if not rules_apriori.empty else 0,
        rules_fpgrowth['lift'].max() if not rules_fpgrowth.empty else 0,
        rules_eclat['lift'].max() if not rules_eclat.empty else 0
    ]
})

print("\nResultados del Benchmarking (Track 1 - Relaciones 1 a 1):")
display(bench_df)

# Mostrar TODAS las reglas ECLAT 1 a 1 ordenadas por Lift y Confianza
rules_eclat_all = rules_eclat.sort_values(by=['lift', 'confidence'], ascending=[False, False])

# Configurar pandas para mostrar todas las filas sin truncar en Jupyter
pd.set_option('display.max_rows', None)

print(f"\n★ Listado Completo de Reglas de Asociación 1 a 1 por Categoría (ECLAT) - Total: {len(rules_eclat_all)} reglas:")
display(rules_eclat_all[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

✓ Librerías analíticas y de minería de datos cargadas correctamente.

--- Cargando Datasets Limpios desde el Pipeline de EDA ---
✓ Matriz Transaccional lista: 11766 facturas x 4 categorías.

--- Ejecutando Track 1: Market Basket Analysis (Relaciones 1 a 1) ---

Resultados del Benchmarking (Track 1 - Relaciones 1 a 1):


,Algoritmo,Tiempo Cómputo (s),Reglas 1-a-1 Generadas,Max Lift
0,Apriori,0.008055,6,1.163372
1,FP-Growth,0.050807,6,1.163372
2,ECLAT,0.012010,6,1.163372



★ Listado Completo de Reglas de Asociación 1 a 1 por Categoría (ECLAT) - Total: 6 reglas:


,antecedents,consequents,support,confidence,lift
5,frozenset({FILTROS}),frozenset({ACEITES}),0.554564,0.878906,1.163372
4,frozenset({ACEITES}),frozenset({FILTROS}),0.554564,0.734053,1.163372
10,frozenset({BATERIAS}),frozenset({FILTROS}),0.019293,0.193686,0.306965
11,frozenset({LLANTAS}),frozenset({FILTROS}),0.021758,0.185105,0.293365
9,frozenset({LLANTAS}),frozenset({ACEITES}),0.021673,0.184382,0.244059
0,frozenset({BATERIAS}),frozenset({ACEITES}),0.017168,0.172355,0.228139
